# What Drives Installs for Auto & Vehicles Apps? A Valvoline App Market Analysis

*Class project · Google Play Store data (~988K apps) · Python, pandas, scikit-learn, Plotly*

**Question.** Valvoline's Instant Oil Change app competes in Google Play's Auto & Vehicles category. What predicts how many installs an app gets there? And do *recurring-use* apps (maintenance trackers, service reminders) keep users more engaged than *one-time* apps (store locators, finders)?

**Approach**
1. Load a cleaned Play Store dataset that joins app metadata with requested permissions.
2. Train Random Forest models to rank which features predict installs: across all apps, within Auto & Vehicles, and using permissions alone.
3. Test how well the category model predicts installs for three named apps it never saw in training: Valvoline, Firestone and Jiffy Lube.
4. Tag Auto & Vehicles apps as Recurring or One-Time and compare their engagement and star ratings.

**Key findings**
- **Model log installs, not raw installs.** On raw install counts, test R² swings from 0.18 to 0.63 depending on the random split, because a handful of apps have 100M+ installs. On log installs the model is stable at R² ≈ 0.81.
- **Rating Count dominates.** It accounts for ~0.89 of feature importance across all apps and ~0.78 within Auto & Vehicles. Permissions barely register, and a permissions-only model explains almost nothing (R² ≈ 0.04).
- **Recurring-use apps engage more.** They collect about twice as many ratings per install as one-time apps (median 0.025 vs 0.012) and are rated higher on average (4.22 vs 3.77 stars).
- **Valvoline's app is well rated.** At 4.40 stars it sits above both group averages; Firestone's app is at 3.50.

**Limitations**
- Feature importance measures association, not cause.
- Recurring/One-Time tags come from keyword matching on app names, and the groups are small (35 and 26 apps). None of the 35 recurring apps requests Calendar access, so the tags could not be cross-checked against permissions.
- "Minimum Installs" is Google Play's install bucket (for example 100,000+), not an exact count.

## 1. Setup
Runs in Google Colab. The dataset is read from Google Drive, and every chart is saved to an `outputs` folder next to it. `kaleido` is the library Plotly uses to export static PNGs.

In [ ]:
!pip install -q "kaleido==0.2.1"

import os
import re

import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

from google.colab import drive
drive.mount("/content/drive")

DATA_PATH = "/content/drive/MyDrive/Colab Notebooks/df_clean.parquet"
OUT_DIR = "/content/drive/MyDrive/Colab Notebooks/outputs"
os.makedirs(OUT_DIR, exist_ok=True)

RANDOM_STATE = 42   # fixed seed so samples, splits and models are repeatable

## 2. Load the data
One row per app (988,310 apps, 44 columns). Each row has Play Store metadata (rating, rating count, size, monetization) plus engineered permission columns: a total count, a count of permission types, and one True/False flag per sensitive permission.

In [ ]:
df = pd.read_parquet(DATA_PATH)
print(df.shape)
print(df["Monetization_Type"].value_counts())
df[["App Name", "Category", "Rating", "Rating Count", "Minimum Installs", "permission_types"]].head()

## 3. Scope: Auto & Vehicles and the three named apps
Valvoline's app is listed under **Auto & Vehicles** (7,403 apps), so the category-level models and charts use only that category. Three named apps serve as reference points: Valvoline (Instant Oil Change), Firestone and Jiffy Lube.

In [ ]:
SCOPE_CATEGORIES = ["Auto & Vehicles"]
scope = df[df["Category"].isin(SCOPE_CATEGORIES)].copy()

print(f"Apps in scope: {len(scope):,}")
print(scope["Rating"].describe().round(2))

In [ ]:
VALVOLINE_ID = "com.mobile.vioc"
FIRESTONE_ID = "com.bsro.fcac"

# Jiffy Lube: find its App Id by name and keep the most-installed match
jiffy_matches = df[df["App Name"].str.contains("jiffy", case=False, na=False)]
display(jiffy_matches[["App Name", "App Id", "Category", "Rating", "Minimum Installs"]])
JIFFY_LUBE_ID = jiffy_matches.sort_values("Minimum Installs", ascending=False)["App Id"].iloc[0]

NAMED_APPS = {
    "Valvoline (Instant Oil Change)": VALVOLINE_ID,
    "Firestone": FIRESTONE_ID,
    "Jiffy Lube": JIFFY_LUBE_ID,
}
named = df[df["App Id"].isin(NAMED_APPS.values())].copy()
named["Label"] = named["App Id"].map({app_id: label for label, app_id in NAMED_APPS.items()})
named[["Label", "App Name", "App Id", "Category", "Rating", "Rating Count", "Minimum Installs"]]

## 4. Modeling helpers
- **Target:** `log_installs`, the log of Minimum Installs (section 5 explains why).
- **Features:** general app attributes (rating, rating count, size, free/ads/in-app purchases, Editors' Choice), permission features, and optionally one-hot category columns.
- **Model:** a Random Forest regressor (100 trees, max depth 10). Tree models need no feature scaling. Each model is scored on a held-out 20% test split.
- **Feature importance** ranks *columns* (Rating Count, perm_Camera, …) by how much they help the model. It shows association, not cause.

In [ ]:
PERM_FLAGS = ["perm_Camera", "perm_Location", "perm_Microphone", "perm_Contacts", "perm_SMS",
              "perm_Phone", "perm_Storage", "perm_Photos", "perm_Calendar"]
PERM_FEATURES = ["permission_count", "num_permission_types", "Sensitive_Access"] + PERM_FLAGS


def build_XY(data, use_general=True, use_permissions=True, use_category=True):
    """Builds the predictor table X and the target y (log installs)."""
    X = pd.DataFrame(index=data.index)
    if use_general:
        X["Rating"] = data["Rating"]
        X["Rating Count"] = data["Rating Count"]
        X["Size_MB"] = data["Size_MB"].fillna(data["Size_MB"].median())
        X["Free"] = data["Free"].astype(int)
        X["Ad Supported"] = data["Ad Supported"].astype(int)
        X["In App Purchases"] = data["In App Purchases"].astype(int)
        X["Editors Choice"] = data["Editors Choice"].astype(int)
    if use_permissions:
        for c in PERM_FEATURES:
            X[c] = data[c].astype(int)
    if use_category:
        X = pd.concat([X, pd.get_dummies(data["Category"], prefix="Category").astype(int)], axis=1)
    return X.fillna(0), data["log_installs"]


def fit_rf(X, y, sample_n=400_000, exclude_index=None):
    """Fits a Random Forest on up to sample_n rows and returns (model, test R², top-20 importances)."""
    if exclude_index is not None:
        X = X.drop(index=exclude_index, errors="ignore")
        y = y.drop(index=exclude_index, errors="ignore")
    if len(X) > sample_n:
        idx = X.sample(n=sample_n, random_state=RANDOM_STATE).index
        X, y = X.loc[idx], y.loc[idx]
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
    model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=RANDOM_STATE, n_jobs=-1)
    model.fit(X_tr, y_tr)
    r2 = r2_score(y_te, model.predict(X_te))
    imp = (pd.DataFrame({"feature": X.columns, "importance": model.feature_importances_})
             .sort_values("importance", ascending=False).head(20))
    print(f"Trained on {len(X_tr):,} rows | Test R²: {r2:.3f}")
    return model, r2, imp


def style(fig, title):
    """Shared title and font styling for every chart."""
    fig.update_layout(
        title=title, title_x=0.5, title_font_size=20,
        xaxis=dict(title_font_size=16, tickfont_size=14),
        yaxis=dict(title_font_size=16, tickfont_size=14),
    )
    return fig


def save(fig, name):
    """Saves a chart as interactive HTML and as a 2800x1800 PNG."""
    fig.write_html(f"{OUT_DIR}/{name}.html")
    fig.write_image(f"{OUT_DIR}/{name}.png", width=1400, height=900, scale=2)


def tier(x):
    if x > 0.05: return "High (>0.05)"
    if x > 0.01: return "Medium (0.01-0.05)"
    return "Low (<0.01)"


def plot_importance(imp, title, r2, name):
    imp = imp.copy()
    imp["tier"] = imp["importance"].apply(tier)
    fig = px.bar(
        imp, x="feature", y="importance", color="tier", log_y=True,
        category_orders={"feature": imp["feature"].tolist()},
        labels={"importance": "Importance Score (Log Scale)", "feature": ""},
        color_discrete_map={"High (>0.05)": "#6C63FF", "Medium (0.01-0.05)": "#F5A623", "Low (<0.01)": "#D0453D"},
    )
    top = imp.iloc[0]
    # Annotations on a log axis use log10 units for y
    fig.add_annotation(x=top["feature"], y=float(np.log10(top["importance"])),
                       text=f"Top Feature: {top['feature']}<br>Importance: {top['importance']:.4f}",
                       showarrow=True, arrowhead=2, ax=60, ay=-40, font=dict(size=18))
    style(fig, f"{title}<br><sup>Random Forest Model (R² = {r2:.3f}) - Top 20 Features</sup>")
    fig.update_layout(xaxis_tickangle=-45, title_font_size=22)
    save(fig, name)
    fig.show()
    return fig

## 5. Why model log installs instead of raw installs
Installs are extremely skewed: the median app has 10,000 installs, but the largest has 10 billion, and 135 apps exceed 100 million. A model trained on raw counts is dominated by those few giants, so its score depends on which of them happen to land in the test split.

In [ ]:
print(df["Minimum Installs"].describe())
print((df["Minimum Installs"] > 1e8).sum(), "apps with 100M+ installs")

**Stability check.** Train the same raw-installs model on five different random splits. Test R² ranges from about 0.18 to 0.63, so the raw model isn't reliable. *(Training on all 988K apps five times takes several minutes.)*

In [ ]:
X1, y1 = build_XY(df)                               # y1 = log installs
y_raw = df.loc[X1.index, "Minimum Installs"]

for rs in [0, 1, 2, 42, 99]:
    X_tr, X_te, y_tr, y_te = train_test_split(X1, y_raw, test_size=0.2, random_state=rs)
    m = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=RANDOM_STATE, n_jobs=-1).fit(X_tr, y_tr)
    print(f"Split seed {rs:>2}: raw-installs test R² = {r2_score(y_te, m.predict(X_te)):.3f}")

**Chart 1a and 1b: raw vs log, full dataset.** With log installs the model is stable (R² ≈ 0.81), and Rating Count clearly dominates. The raw model spreads importance across permission counts, which is noise from the outliers rather than a real signal.

In [ ]:
model_raw, r2_raw, imp_raw = fit_rf(X1, y_raw, sample_n=len(X1))
fig_raw = plot_importance(imp_raw, "Raw Installs Produce an Unstable, Unreliable Model",
                          r2_raw, "chart1a_raw_installs_importance")

In [ ]:
model_log, r2_log, imp_log = fit_rf(X1, y1, sample_n=len(X1))
fig_log = plot_importance(imp_log, "Log-Transforming Installs Produces a Stable, Reliable Model",
                          r2_log, "chart1b_log_installs_importance")

## 6. What drives installs within Auto & Vehicles
**Chart 2: category model.** Same features, Auto & Vehicles apps only. The three named apps are held out of training so that the predictions in section 7 are for apps the model has never seen. Category columns are dropped because a single category adds no information.

In [ ]:
holdout_index = named.index
multi_category = len(SCOPE_CATEGORIES) > 1

X2, y2 = build_XY(scope, use_category=multi_category)
model2, r2_2, imp2 = fit_rf(X2, y2, exclude_index=holdout_index)
fig2 = plot_importance(imp2, "Rating Count and Rating Drive Installs — Permissions Barely Register",
                       r2_2, "chart2_category_importance")

**Chart 3: permissions-only model.** Rating Count can drown out every other signal, so this model uses only the permission columns. Its low R² is the finding: what an app asks permission for says very little about how many installs it gets.

In [ ]:
X3, y3 = build_XY(scope, use_general=False, use_permissions=True, use_category=False)
model3, r2_3, imp3 = fit_rf(X3, y3, exclude_index=holdout_index)
fig3 = plot_importance(imp3, "Which Permission Factors Correlate With Installs? (Auto & Vehicles)",
                       r2_3, "chart3_permissions_importance")

## 7. Predicting installs for the named apps
The category model (Chart 2) predicts installs for Valvoline, Firestone and Jiffy Lube, none of which it saw in training. A ratio near 1.0 means the app is performing about as well as its ratings, size and permissions would suggest.

In [ ]:
named_X, _ = build_XY(named, use_category=multi_category)
named_X = named_X.reindex(columns=X2.columns, fill_value=0)

predictions = pd.DataFrame({
    "App": named["Label"],
    "Predicted installs": np.expm1(model2.predict(named_X)).round(),
    "Actual installs": named["Minimum Installs"],
})
predictions["Predicted / actual"] = (predictions["Predicted installs"] / predictions["Actual installs"]).round(2)
predictions.set_index("App")

## 8. Supporting charts: privacy policies and monetization

In [ ]:
gap = scope.groupby("Sensitive_Access")["Has_Privacy_Policy"].mean().mul(100).reset_index()
gap["Sensitive_Access"] = gap["Sensitive_Access"].map({True: "Requests camera/mic/location", False: "Does not"})

fig5 = px.bar(gap, x="Sensitive_Access", y="Has_Privacy_Policy",
              labels={"Has_Privacy_Policy": "% With Privacy Policy", "Sensitive_Access": ""})
style(fig5, "Privacy Policy Rate by Sensor Access (Auto & Vehicles)")
save(fig5, "chart5_privacy_gap")
fig5.show()

In [ ]:
top_rated = scope[scope["Rating"] >= 4.0]
mix = top_rated["Monetization_Type"].value_counts(normalize=True).mul(100).reset_index()
mix.columns = ["Monetization_Type", "Percent"]

fig6 = px.bar(mix, x="Monetization_Type", y="Percent")
fig6.update_traces(texttemplate="%{y:.1f}%", textposition="outside")
style(fig6, "Monetization Mix of 4.0★+ Auto & Vehicles Apps")
fig6.update_layout(xaxis_title="", xaxis_categoryorder="total descending", yaxis_title="% of 4.0★+ Rated Apps")
save(fig6, "chart6_monetization_mix")
fig6.show()

## 9. Recurring vs one-time apps
**Tagging.** Each Auto & Vehicles app is tagged by keyword patterns in its name:
- **Recurring:** apps people return to, such as maintenance trackers, service reminders and mileage or fuel logs.
- **One-Time:** apps people open for a single task, such as store locators, finders and directories.

Apps whose names match unrelated topics (racing games, dash cams, radio, insurance quotes, …) are excluded, and apps that match neither list are left untagged.

In [ ]:
RECURRING_PATTERNS = [
    r"\bmaintenance track(er|ing)?\b",
    r"\bservice remind(er)?\b",
    r"\boil change remind(er)?\b",
    r"\bmileage log(s|ger|book|ging)?\b",
    r"\bvehicle log(s|ger|book|ging)?\b",
    r"\bcar log(s|ger|book|ging)?\b",
    r"\bmaintenance log(s|ger|book|ging)?\b",
    r"\bfuel log(s|ger|book|ging)?\b",
    r"\bservice due\b",
    r"\brepair track(er)?\b",
]

ONE_TIME_PATTERNS = [
    r"\blocator\b", r"\bfinder\b", r"\bnear me\b", r"\bdirectory\b",
    r"\bstore locator\b", r"\bdealer locator\b", r"\bmap\b",
]

EXCLUDE_PATTERNS = [
    r"\brace\b", r"\bdrift\b", r"\bgps navigat\w*\b", r"\bdash ?cam\b",
    r"\bradio\b", r"\bmusic\b", r"\bvideo\b", r"\bparking meter\b",
    r"\bfuel price\b", r"\binsurance quote\b",
]


def tag_utility_type(name):
    name_lower = str(name).lower()
    if any(re.search(p, name_lower) for p in EXCLUDE_PATTERNS):
        return None
    if any(re.search(p, name_lower) for p in RECURRING_PATTERNS):
        return "Recurring"
    if any(re.search(p, name_lower) for p in ONE_TIME_PATTERNS):
        return "One-Time"
    return None


scope["Utility_Type"] = scope["App Name"].apply(tag_utility_type)
tagged = scope[scope["Utility_Type"].notna()].copy()
print(tagged["Utility_Type"].value_counts())

# Save the tagged list so the tags can be reviewed by hand
tagged[["App Name", "Utility_Type", "Rating Count", "Minimum Installs"]].sort_values("Utility_Type") \
    .to_csv(f"{OUT_DIR}/utility_type_review.csv", index=False)

**Cross-check with permissions.** If the recurring tags were right, some of those apps should request Calendar access for reminders. None of them do, so the tags rest on app names alone (see Limitations).

In [ ]:
recurring = tagged["Utility_Type"] == "Recurring"
print(f"Recurring apps: {recurring.sum()}")
print(f"Recurring apps requesting Calendar access: {(recurring & tagged['perm_Calendar']).sum()}")

**Engagement and ratings.** *Engagement ratio* = Rating Count ÷ Minimum Installs, the share of installers who bothered to leave a rating. Apps with zero installs are dropped to avoid dividing by zero.

In [ ]:
tagged = tagged[tagged["Minimum Installs"] > 0].copy()
tagged["engagement_ratio"] = tagged["Rating Count"] / tagged["Minimum Installs"]

print(tagged.groupby("Utility_Type")["engagement_ratio"].describe().round(4))
print()
print(tagged.groupby("Utility_Type")["Rating"].agg(["mean", "median", "count"]).round(2))

In [ ]:
# Valvoline and Firestone are overlaid as stars on the final charts
OVERLAYS = [  # (label, App Id, group to plot it in, marker color)
    ("Firestone", FIRESTONE_ID, "Recurring", "black"),
    ("Valvoline (Instant Oil Change)", VALVOLINE_ID, "One-Time", "red"),
]

for label, app_id, _, _ in OVERLAYS:
    r = df[df["App Id"] == app_id].iloc[0]
    print(f"{label}: Rating = {r['Rating']:.2f} | Rating Count = {r['Rating Count']:,.0f} | "
          f"Installs = {r['Minimum Installs']:,.0f} | Engagement ratio = {r['Rating Count'] / r['Minimum Installs']:.4f}")

**Slide 9: star ratings.** Recurring apps are rated higher on average than one-time apps.

In [ ]:
group_means = tagged.groupby("Utility_Type")["Rating"].mean()

fig_ratings = px.box(tagged, x="Utility_Type", y="Rating", color="Utility_Type",
                     points="all", hover_name="App Name", labels={"Utility_Type": ""})

for group in ["Recurring", "One-Time"]:
    fig_ratings.add_annotation(
        x=group, y=group_means[group] + 0.2,
        text=f"Category avg: {group_means[group]:.2f}",
        showarrow=False, font=dict(color="black"),
        bgcolor="white", bordercolor="black", borderwidth=1,
    )

for label, app_id, group, color in OVERLAYS:
    r = df[df["App Id"] == app_id].iloc[0]
    fig_ratings.add_scatter(
        x=[group], y=[r["Rating"]], mode="markers+text",
        marker=dict(size=16, color=color, symbol="star"),
        text=[f"{label}: {r['Rating']:.2f}"], textposition="bottom center",
        textfont=dict(size=14), showlegend=False,
    )

style(fig_ratings, "Star Ratings Confirm the Engagement Gap Is Real")
fig_ratings.update_annotations(font_size=14)
save(fig_ratings, "chart7_star_ratings")
fig_ratings.show()

**Slide 10: engagement intensity.** Shown on a log scale because ratios range from under 0.01 to over 0.5.

In [ ]:
fig_engagement = px.strip(tagged, x="Utility_Type", y="engagement_ratio", color="Utility_Type",
                          hover_name="App Name",
                          hover_data={"Rating Count": True, "Minimum Installs": ":,", "Utility_Type": False},
                          labels={"engagement_ratio": "Rating Count per Install", "Utility_Type": ""})
fig_engagement.update_yaxes(type="log")

for group in ["Recurring", "One-Time"]:
    fig_engagement.add_annotation(
        x=group, xref="x", yref="paper", y=1.05,
        text=f"Avg Rating: {group_means[group]:.2f}",
        showarrow=False, font=dict(color="black"),
        bgcolor="white", bordercolor="black", borderwidth=1,
    )

for label, app_id, group, color in OVERLAYS:
    r = df[df["App Id"] == app_id].iloc[0]
    fig_engagement.add_scatter(
        x=[group], y=[r["Rating Count"] / r["Minimum Installs"]], mode="markers+text",
        marker=dict(size=16, color=color, symbol="star"),
        text=[f"{label} (Rating: {r['Rating']:.2f})"], textposition="top center",
        textfont=dict(size=14), showlegend=False,
    )

style(fig_engagement, "Recurring Apps Show Deeper Engagement — And Higher Ratings, Too")
fig_engagement.update_annotations(font_size=14)
save(fig_engagement, "chart8_engagement_and_ratings")
fig_engagement.show()

## 10. Export interactive charts for the presentation
Full-page, responsive HTML versions of the slide 9 and slide 10 charts, published with GitHub Pages and linked from the slides.

In [ ]:
for fig, name in [(fig_ratings, "slide9_star_ratings.html"),
                  (fig_engagement, "slide10_engagement_and_ratings.html")]:
    fig.write_html(f"{OUT_DIR}/{name}",
                   include_plotlyjs=True,
                   full_html=True,
                   default_width="100%",
                   default_height="100vh",
                   config={"responsive": True, "displaylogo": False})

print(f"Saved to {OUT_DIR}:")
for f in sorted(os.listdir(OUT_DIR)):
    print("  ", f)